In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

In [30]:
from pathlib import Path
from io import BytesIO, StringIO
import re
import pandas as pd

In [24]:
DATA_DIR = Path(r"C:\ai\source\09_project\data")   
file_names = [
    "대여소별 이용정보 22.1.csv",
    "대여소별 이용정보 22.2.csv",
    "대여소별 이용정보 22.3.csv",
    "대여소별 이용정보 22.4.csv",
    "대여소별 이용정보 22.5.csv",
    "대여소별 이용정보 22.6.csv",
    "서울특별시 공공자전거 대여소별 이용정보(월별)_22.7-22.12.csv",
]

FINAL_COLUMNS = ["자치구", "대여년월", "대여소 명", "대여건수"]


In [42]:
def load_file(file_path):
    
    raw = file_path.read_bytes()

    # 엑셀 파일 처리
    if raw.startswith(b"PK\x03\x04"):
        df = pd.read_excel(
            BytesIO(raw),
            engine="openpyxl",
            dtype="string"
        )

    # CSV 파일 처리
    else:
        text = None

        for encoding in ["utf-8-sig", "cp949", "euc-kr"]:
            try:
                text = raw.decode(encoding)
                break
            except UnicodeDecodeError:
                continue

        if text is None:
            raise ValueError(
                f"인코딩 확인 불가: {file_path.name}"
            )

        lines = text.splitlines()

        header_index = next(
            (
                i for i, line in enumerate(lines)
                if "대여소 그룹" in line and "대여소 명" in line
            ),
            None
        )

        if header_index is None:
            raise ValueError(
                f"표의 헤더를 찾지 못했습니다: {file_path.name}"
            )

        df = pd.read_csv(
            StringIO(text),
            skiprows=header_index,
            dtype="string"
        )

        metadata = "\n".join(lines[:header_index])

        match = re.search(
            r"(\d{6})\s*~\s*(\d{6})",
            metadata
        )

        if match is None or match.group(1) != match.group(2):
            raise ValueError(
                f"단일 월 확인 불가: {file_path.name}"
            )

        df["대여년월"] = match.group(1)
    
   
    
    df. columns = df.columns.str.strip()
    df = df.rename(columns={
        "대여소 그룹":"자치구",
        "일자/월":"대여년월",
        "일/월":"대여년월",
        "기준년월":"대여년월",
        "대여소명":"대여소 명",
        "대여 건수":"대여건수",
    })
    missing = set(FINAL_COLUMNS)-set(df.columns)

    if missing:
        raise ValueError(
            f"{file_path.name}:필요한 컬럼 누락 {missing}"        
        )
    df = df[FINAL_COLUMNS].copy()
    for col in FINAL_COLUMNS:
        df[col] = df[col].astype("string").str.strip()

    df = df.replace(r"^\s*$",pd.NA,regex=True)

    df = df.dropna(how="all").copy()


    if df.isna().any().any():
        raise ValueError(
            f"{file_path.name}: 결측치 확인 필요\n"
            f"{df.isna().sum()}"
        )
    month_text = df["대여년월"].str.replace(
            r"\.0+$", "", regex=True
        )   
    if not month_text.str.fullmatch(r"\d{6}").all():
        raise ValueError(
            f"{file_path.name}:년월 형식 오류"
        )
    month_dates = pd.to_datetime(
        month_text,
        format="%Y%m",
        errors="raise"
    )    

    if not month_dates.dt.year.eq(2022).all():
        raise ValueError(
            f"{file_path.name}:2022년 외 자료있음"
        )
    df["대여년월"] = (
        month_dates.dt.strftime("%Y%m").astype("string")
    )

    counts = pd.to_numeric(
        df["대여건수"].str.replace(",","",regex=False),
        errors="raise"
    )

    invalid_counts = (
        counts.isna()
        | counts.lt(0)
        | counts.mod(1).ne(0)
    )
    if invalid_counts.any():
            raise ValueError(
                f"{file_path.name}: 대여건수에 결측/음수/비정수 존재"
            )

    df["대여건수"] = counts.astype("int64")

    return df        

In [43]:
missing_files = [
    name
    for name in file_names
    if not (DATA_DIR / name).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"폴더: {DATA_DIR.resolve()}\n"
        f"없는 파일: {missing_files}"
    )

In [44]:
dataframes = []

for file_name in file_names:
    file_path = DATA_DIR / file_name

    
    df = load_file(file_path)

    
    dataframes.append(df)

    print(f"{file_name}: {len(df):,}행")


df_total = pd.concat(
    dataframes,
    ignore_index=True
)

대여소별 이용정보 22.1.csv: 2,588행
대여소별 이용정보 22.2.csv: 2,594행
대여소별 이용정보 22.3.csv: 2,607행
대여소별 이용정보 22.4.csv: 2,602행
대여소별 이용정보 22.5.csv: 2,616행
대여소별 이용정보 22.6.csv: 2,627행
서울특별시 공공자전거 대여소별 이용정보(월별)_22.7-22.12.csv: 16,019행


In [45]:
key_columns = ["자치구", "대여년월", "대여소 명"]

duplicate_mask = df_total.duplicated(
    subset=key_columns,
    keep=False
)

if duplicate_mask.any():
    print(
        df_total.loc[duplicate_mask]
        .sort_values(key_columns)
        .to_string(index=False)
    )

    raise ValueError(
        "같은 자치구/월/대여소가 중복됩니다. 원본 확인이 필요합니다."
    )

# 2022년 1~12월 자료가 모두 있는지 확인
expected_months = {
    f"2022{month:02d}"
    for month in range(1, 13)
}

if set(df_total["대여년월"]) != expected_months:
    raise ValueError(
        "2022년 1~12월 중 빠진 월 또는 예상 밖의 월이 있습니다."
    )

In [46]:
df_total = df_total.sort_values(
    by=["대여년월", "자치구", "대여소 명"]
).reset_index(drop=True)


df_total = df_total[FINAL_COLUMNS]

print("최종 크기:", df_total.shape)

print("\n자료형:")
print(df_total.dtypes)

print("\n결측치:")
print(df_total.isna().sum())

print("\n앞부분:")
print(df_total.head())

print("\n월별 행 수와 대여건수 합계:")
print(
    df_total.groupby("대여년월")["대여건수"]
    .agg(["size", "sum"])
)

print("\n전체 대여건수:")
print(f"{df_total['대여건수'].sum():,}")

print("\n대여건수가 0인 행 수:")
print(df_total["대여건수"].eq(0).sum())

최종 크기: (31653, 4)

자료형:
자치구      string
대여년월     string
대여소 명    string
대여건수      int64
dtype: object

결측치:
자치구      0
대여년월     0
대여소 명    0
대여건수     0
dtype: int64

앞부분:
   자치구    대여년월                           대여소 명  대여건수
0  강남구  202201                2301. 현대고등학교 건너편   495
1  강남구  202201  2302. 교보타워 버스정류장(신논현역 3번출구 후면)   489
2  강남구  202201                 2303. 논현역 10번출구   413
3  강남구  202201                    2304. 대현그린타워    11
4  강남구  202201              2305. MCM 본사 직영점 앞   163

월별 행 수와 대여건수 합계:
        size      sum
대여년월                 
202201  2588  1420013
202202  2594  1349521
202203  2607  2424115
202204  2602  3991127
202205  2616  4963610
202206  2627  4257539
202207  2647  4101127
202208  2656  3620585
202209  2664  4760532
202210  2678  4470269
202211  2679  3834831
202212  2695  1756715

전체 대여건수:
40,949,984

대여건수가 0인 행 수:
3


In [49]:
df_total.head(20)

,자치구,대여년월,대여소 명,대여건수
0,강남구,202201,2301. 현대고등학교 건너편,495
1,강남구,202201,2302. 교보타워 버스정류장(신논현역 3번출구 후면),489
2,강남구,202201,2303. 논현역 10번출구,413
3,강남구,202201,2304. 대현그린타워,11
4,강남구,202201,2305. MCM 본사 직영점 앞,163
5,강남구,202201,2306. 압구정역 2번 출구 옆,934
6,강남구,202201,2307. 압구정 한양 3차 아파트,119
7,강남구,202201,2308. 압구정파출소 앞,331
8,강남구,202201,2309. 청담역(우리들병원 앞),166
9,강남구,202201,2310. 청담동 맥도날드 옆(위치),153


In [50]:
output_path = DATA_DIR / "공공자전거_대여소별_이용정보_2022_전처리.csv"

df_total.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path.resolve())

저장 완료: C:\ai\source\09_project\data\공공자전거_대여소별_이용정보_2022_전처리.csv
